# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [9]:
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import gutenberg
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, HTML

# Asegurar descarga de Gutenberg
nltk.download('gutenberg', quiet=True)

# 1. Seleccionar 10 documentos del corpus Project Gutenberg
file_ids = gutenberg.fileids()[:10]

corpus_docs = []
for file_id in file_ids:
    text = gutenberg.raw(file_id)
    title = file_id.replace('.txt', '').replace('-', ' ').title()
    corpus_docs.append({'id': file_id, 'title': title, 'raw_text': text})

# Preprocesador eficiente (Regex idéntico a TfidfVectorizer, evita la lentitud de word_tokenize)
def preprocess(text):
    return re.findall(r'(?u)\b\w+\b', text.lower())

# Procesar tokens por documento
for doc in corpus_docs:
    doc['tokens'] = preprocess(doc['raw_text'])
    doc['unique_terms'] = len(set(doc['tokens']))
    doc['length'] = len(doc['tokens'])

# 2. Tabla de Metadatos estilizada
df_meta = pd.DataFrame([
    {
        'Identificador': d['id'],
        'Título': d['title'],
        'Términos Únicos': f"{d['unique_terms']:,}"
    }
    for d in corpus_docs
])

# 3. Estadísticas del corpus global
doc_lengths = [d['length'] for d in corpus_docs]
vocab_global = set(term for d in corpus_docs for term in d['tokens'])

df_resumen = pd.DataFrame([
    {'Métrica': 'Número de documentos', 'Valor': f"{len(corpus_docs)}"},
    {'Métrica': 'Tamaño del vocabulario global', 'Valor': f"{len(vocab_global):,} términos"},
    {'Métrica': 'Longitud mínima', 'Valor': f"{np.min(doc_lengths):,} términos"},
    {'Métrica': 'Longitud media', 'Valor': f"{np.mean(doc_lengths):,.2f} términos"},
    {'Métrica': 'Longitud máxima', 'Valor': f"{np.max(doc_lengths):,} términos"}
])

# Visualización limpia en HTML/Jupyter
display(HTML("<h3>Metadatos de Documentos (Top 10)</h3>"))
display(df_meta)

display(HTML("<h3>Resumen Estadístico del Corpus</h3>"))
display(df_resumen)

,Identificador,Título,Términos Únicos
0,austen-emma.txt,Austen Emma,"7,256"
1,austen-persuasion.txt,Austen Persuasion,"5,777"
2,austen-sense.txt,Austen Sense,"6,336"
3,bible-kjv.txt,Bible Kjv,"12,744"
4,blake-poems.txt,Blake Poems,"1,512"
5,bryant-stories.txt,Bryant Stories,"3,900"
6,burgess-busterbrown.txt,Burgess Busterbrown,"1,539"
7,carroll-alice.txt,Carroll Alice,"2,571"
8,chesterton-ball.txt,Chesterton Ball,"8,267"
9,chesterton-brown.txt,Chesterton Brown,"7,739"


,Métrica,Valor
0,Número de documentos,10
1,Tamaño del vocabulario global,"25,402 términos"
2,Longitud mínima,"6,936 términos"
3,Longitud media,"147,447.60 términos"
4,Longitud máxima,"854,046 términos"


Se recuperaron 10 documentos del corpus Project Gutenberg utilizando la biblioteca NLTK. Se conservaron los textos originales y se realizó un preprocesamiento que convierte el texto a minúsculas y separa las palabras mediante expresiones regulares. Posteriormente, se calculó la cantidad de términos únicos y la longitud de cada documento. Finalmente, se obtuvo un resumen estadístico del corpus, identificando 25.402 términos únicos, una longitud mínima de 6.936 términos, una media de 147.447,60 y una máxima de 854.046 términos.

### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [10]:
def calculate_tf(term, tokens):
    """Calcula la frecuencia del término (TF) en un documento."""
    return tokens.count(term)

def calculate_df(term, corpus_tokens):
    """Calcula el Document Frequency (DF) de un término en todo el corpus."""
    return sum(1 for tokens in corpus_tokens if term in set(tokens))

def calculate_idf(df, N):
    """Calcula el IDF según la fórmula de scikit-learn con smooth_idf=False."""
    return math.log(N / df) + 1.0

def calculate_tfidf(tf, idf):
    """Calcula el peso TF-IDF."""
    return tf * idf

Se definieron cuatro funciones para realizar los cálculos fundamentales del modelo TF-IDF. La primera calcula cuántas veces aparece un término en un documento (TF); la segunda determina en cuántos documentos aparece dicho término (DF); la tercera calcula su frecuencia inversa documental (IDF); y la cuarta multiplica TF por IDF para obtener el peso de importancia de cada término.

### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [11]:
# 1. Vocabulario ordenado
vocab = sorted(list(set(t for d in corpus_docs for t in d['tokens'])))
word2idx = {w: i for i, w in enumerate(vocab)}

N = len(corpus_docs)
all_doc_tokens = [d['tokens'] for d in corpus_docs]

# 2. Matriz TF
tf_matrix = np.zeros((N, len(vocab)))
for i, d in enumerate(corpus_docs):
    counts = pd.Series(d['tokens']).value_counts()
    for word, count in counts.items():
        if word in word2idx:
            tf_matrix[i, word2idx[word]] = count

# 3. Document Frequency (DF) optimizado
df_vector = np.sum(tf_matrix > 0, axis=0)

# 4. Inverse Document Frequency (IDF)
idf_vector = np.log(N / df_vector) + 1.0

# 5. Matriz TF-IDF explícita
tfidf_explicit = tf_matrix * idf_vector
print(f"Forma de la matriz TF-IDF explícita: {tfidf_explicit.shape}")

Forma de la matriz TF-IDF explícita: (10, 25402)


Se construyó un vocabulario global ordenado con todos los términos únicos del corpus y se generó la matriz de frecuencias TF. Posteriormente, se calculó la frecuencia documental (DF) y el valor IDF de cada término para obtener la matriz TF-IDF mediante multiplicación. Como resultado, se obtuvo una matriz de dimensiones (10, 25402), que representa los 10 documentos mediante los 25.402 términos del vocabulario.

### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [14]:
# TfidfVectorizer requiere tokens como cadenas unidas si ya preprocesamos
docs_as_str = [" ".join(d['tokens']) for d in corpus_docs]

vectorizer = TfidfVectorizer(
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
    analyzer=lambda x: x.split() # Usa exactamente nuestros tokens
)

tfidf_sklearn = vectorizer.fit_transform(docs_as_str).toarray()

# Comprobar si el orden del vocabulario coincide
feature_names = vectorizer.get_feature_names_out()
assert list(feature_names) == vocab, "El vocabulario no está alineado."

# Máxima diferencia absoluta
max_abs_diff = np.max(np.abs(tfidf_explicit - tfidf_sklearn))
print(f"Máxima diferencia absoluta: {max_abs_diff:.10e}")

if max_abs_diff < 1e-7:
    print(" Verificación exitosa: Las matrices de TF-IDF explícita y Scikit-Learn coinciden perfectamente.")
else:
    print(" Hay una discrepancia entre las matrices.")

Máxima diferencia absoluta: 0.0000000000e+00
 Verificación exitosa: Las matrices de TF-IDF explícita y Scikit-Learn coinciden perfectamente.


Se utilizó la herramienta TfidfVectorizer de Scikit-Learn con los mismos parámetros y términos empleados en la implementación manual. Posteriormente, se compararon ambas matrices TF-IDF calculando su máxima diferencia absoluta. El resultado fue 0.0000000000, lo que demuestra que ambas implementaciones generan exactamente los mismos valores y que los cálculos realizados manualmente son correctos.

### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [16]:
# 1. Definición de las cuatro consultas y necesidad de información
query_definitions = [
    {
        "consulta": "sin and punishment in human nature",
        "necesidad": "Encontrar documentos relacionados con el pecado, el castigo y el comportamiento humano."
    },
    {
        "consulta": "love, marriage and romantic relationships",
        "necesidad": "Encontrar textos que traten sobre el amor, el matrimonio y las relaciones sentimentales."
    },
    {
        "consulta": "sea sailing, ships and naval adventures",
        "necesidad": "Encontrar relatos o documentos relacionados con el mar, la navegación y los barcos."
    },
    {
        "consulta": "divine wisdom, god and religious faith",
        "necesidad": "Encontrar contenido relacionado con Dios, la sabiduría divina y la fe religiosa."
    }
]
queries = [item["consulta"] for item in query_definitions]

def vectorize_query_explicit(q_str, vocab, word2idx, idf_vec):
    """Construye el vector TF-IDF de la consulta con el vocabulario e IDF del corpus."""
    q_tokens = preprocess(q_str)
    q_tf = np.zeros(len(vocab))
    for term in q_tokens:
        if term in word2idx:
            q_tf[word2idx[term]] += 1
    return q_tf * idf_vec

def cosine_similarity_explicit(vec1, vec2):
    """Calcula la similitud coseno mediante producto punto y normas."""
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(np.dot(vec1, vec2) / (norm1 * norm2))

# Mostrar qué busca cada consulta
display(pd.DataFrame([
    {"Consulta": x["consulta"], "Necesidad de información": x["necesidad"]}
    for x in query_definitions
]))

# --- Desglose de similitud para la consulta 1 y el documento 1 ---
q_idx = 0
doc_idx = 0
q1 = queries[q_idx]

# IMPORTANTE: vocab es la lista ordenada que corresponde a las columnas de las matrices.
q1_vec = vectorize_query_explicit(q1, vocab, word2idx, idf_vector)
d0_vec = tfidf_explicit[doc_idx]
doc_title = corpus_docs[doc_idx]["title"]

shared_indices = np.where((q1_vec > 0) & (d0_vec > 0))[0]

dot_prod = float(np.dot(q1_vec, d0_vec))
norm_q = float(np.linalg.norm(q1_vec))
norm_d = float(np.linalg.norm(d0_vec))
sim_cos = cosine_similarity_explicit(q1_vec, d0_vec)

q1_sklearn = vectorizer.transform([" ".join(preprocess(q1))]).toarray()[0]
sim_cos_sklearn = cosine_similarity([q1_sklearn], [tfidf_sklearn[doc_idx]])[0][0]
diff_abs = abs(sim_cos - sim_cos_sklearn)

if len(shared_indices) > 0:
    df_shared = pd.DataFrame({
        "Término": [vocab[idx] for idx in shared_indices],
        "TF-IDF Consulta (q_i)": [q1_vec[idx] for idx in shared_indices],
        "TF-IDF Documento (d_i)": [d0_vec[idx] for idx in shared_indices],
        "Producto Parcial (q_i × d_i)": [q1_vec[idx] * d0_vec[idx] for idx in shared_indices]
    })
    display(df_shared.style.format({
        "TF-IDF Consulta (q_i)": "{:.4f}",
        "TF-IDF Documento (d_i)": "{:.4f}",
        "Producto Parcial (q_i × d_i)": "{:.4f}"
    }).hide(axis="index"))
else:
    print("No existen términos compartidos entre la consulta y el documento.")

df_metrics = pd.DataFrame([
    {"Métrica": "Producto punto (q · d)", "Valor": f"{dot_prod:.6f}"},
    {"Métrica": "Norma de consulta (||q||)", "Valor": f"{norm_q:.6f}"},
    {"Métrica": "Norma de documento (||d||)", "Valor": f"{norm_d:.6f}"},
    {"Métrica": "Similitud coseno explícita", "Valor": f"{sim_cos:.6f}"},
    {"Métrica": "Similitud coseno Scikit-Learn", "Valor": f"{sim_cos_sklearn:.6f}"},
    {"Métrica": "Diferencia absoluta", "Valor": f"{diff_abs:.2e}"}
])
display(df_metrics)


,Consulta,Necesidad de información
0,sin and punishment in human nature,Encontrar documentos relacionados con el pecad...
1,"love, marriage and romantic relationships","Encontrar textos que traten sobre el amor, el ..."
2,"sea sailing, ships and naval adventures",Encontrar relatos o documentos relacionados co...
3,"divine wisdom, god and religious faith","Encontrar contenido relacionado con Dios, la s..."


Término,TF-IDF Consulta (q_i),TF-IDF Documento (d_i),Producto Parcial (q_i × d_i)
and,1.0000,4896.0000,4896.0000
human,1.2231,13.4546,16.4569
in,1.0000,2188.0000,2188.0000
nature,1.1054,45.3198,50.0947
punishment,1.6931,6.7726,11.4670
sin,1.9163,1.9163,3.6722


,Métrica,Valor
0,Producto punto (q · d),7165.690738
1,Norma de consulta (||q||),3.355118
2,Norma de documento (||d||),14995.189264
3,Similitud coseno explícita,0.142429
4,Similitud coseno Scikit-Learn,0.142429
5,Diferencia absoluta,8.33e-17


Se definieron cuatro consultas relacionadas con diferentes temáticas y se transformaron en vectores TF-IDF utilizando el vocabulario y los valores IDF del corpus. Después, se implementó la similitud coseno mediante el producto punto y las normas de los vectores. Al comparar la primera consulta con el documento Austen Emma, se obtuvo una similitud de 0.142429, coincidiendo prácticamente con el resultado de Scikit-Learn. Esto permite comprobar que el cálculo de similitud funciona correctamente.

### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [19]:
# Ranking TF-IDF para las cuatro consultas

def get_excerpt(raw_text, query, window=180):
    """Devuelve un fragmento del texto original alrededor del primer término coincidente."""
    terms = [t for t in preprocess(query) if t in vocab_global]
    text_lower = raw_text.lower()
    for term in terms:
        match = re.search(r"\b" + re.escape(term) + r"\b", text_lower)
        if match:
            start = max(0, match.start() - window // 2)
            end = min(len(raw_text), match.end() + window // 2)
            return " ".join(raw_text[start:end].split()).replace("\n", " ") + "..."
    return "No se encontró un fragmento coincidente."

ranking_tfidf = {}

for q_info in query_definitions:
    query = q_info["consulta"]

    # Representación explícita de la consulta con el IDF del corpus
    q_vec = vectorize_query_explicit(query, vocab, word2idx, idf_vector)
    scores_explicit = np.array([
        cosine_similarity_explicit(q_vec, tfidf_explicit[i])
        for i in range(N)
    ])

    # Verificación con Scikit-Learn
    q_vec_sklearn = vectorizer.transform([" ".join(preprocess(query))])
    scores_sklearn = cosine_similarity(q_vec_sklearn, tfidf_sklearn).flatten()

    max_diff = float(np.max(np.abs(scores_explicit - scores_sklearn)))
    print(f"\nConsulta: {query}")
    print(f"Necesidad: {q_info['necesidad']}")
    print(f"Diferencia máxima entre rankings: {max_diff:.12e}")

    order = np.argsort(scores_explicit)[::-1][:5]
    rows = []
    for position, idx in enumerate(order, start=1):
        rows.append({
            "Posición": position,
            "Identificador": corpus_docs[idx]["id"],
            "Título": corpus_docs[idx]["title"],
            "Puntuación": scores_explicit[idx],
            "Fragmento": get_excerpt(corpus_docs[idx]["raw_text"], query)
        })

    ranking_tfidf[query] = pd.DataFrame(rows)
    display(ranking_tfidf[query].style.format({"Puntuación": "{:.6f}"}).hide(axis="index"))



Consulta: sin and punishment in human nature
Necesidad: Encontrar documentos relacionados con el pecado, el castigo y el comportamiento humano.
Diferencia máxima entre rankings: 1.998401444325e-15


Posición,Identificador,Título,Puntuación,Fragmento
1,blake-poems.txt,Blake Poems,0.212743,[Poems by William Blake 1789] SONGS OF INNOCENCE AND OF EXPERIENCE and THE BOOK of THEL SONGS OF INNOCENCE INTRODUCTION Piping down...
2,bible-kjv.txt,Bible Kjv,0.183942,"fallen? 4:7 If thou doest well, shalt thou not be accepted? and if thou doest not well, sin lieth at the door. And unto thee shall be his desire, and thou shalt rule over him. 4:8..."
3,bryant-stories.txt,Bryant Stories,0.157996,"en that I ken, Full of little gentlemen; Little caps of blue they wear, And green ribbons, very fair. (Flax.) From house to house he goes,..."
4,austen-persuasion.txt,Austen Persuasion,0.157950,"t, never took up any book but the Baronetage; there he found occupation for an idle hour, and consolation in a distressed one; there his faculties were roused into admiration and resp..."
5,chesterton-ball.txt,Chesterton Ball,0.149194,"ut any romantic nonsense about honour or anything of that sort. Is not bloodshed a great sin?"" ""No,"" said MacIan, speaking for the first time. ""Well, really, really!"" said the peac..."



Consulta: love, marriage and romantic relationships
Necesidad: Encontrar textos que traten sobre el amor, el matrimonio y las relaciones sentimentales.
Diferencia máxima entre rankings: 1.998401444325e-15


Posición,Identificador,Título,Puntuación,Fragmento
1,blake-poems.txt,Blake Poems,0.196923,"ay. ""And we are put on earth a little space, That we may learn to bear the beams of love And these black bodies and this sunburnt face Is but a cloud, and like a shady grove...."
2,bible-kjv.txt,Bible Kjv,0.181762,"d go out to the field, and take me some venison; 27:4 And make me savoury meat, such as I love, and bring it to me, that I may eat; that my soul may bless thee before I die. 27:5 And..."
3,bryant-stories.txt,Bryant Stories,0.152811,"onderful light glowed from her heart, the sound of thunder rolled through the sky, and a love greater than words can tell filled the Cloud; down, down, close to the earth she swept,..."
4,austen-persuasion.txt,Austen Persuasion,0.136246,"reserve. Lady Russell, indeed, had scarcely any influence with Elizabeth, and seemed to love her, rather because she would love her, than because Elizabeth deserved it. She had neve..."
5,austen-emma.txt,Austen Emma,0.127094,"introduced him to Miss Churchill, of a great Yorkshire family, and Miss Churchill fell in love with him, nobody was surprized, except her brother and his wife, who had never seen him,..."



Consulta: sea sailing, ships and naval adventures
Necesidad: Encontrar relatos o documentos relacionados con el mar, la navegación y los barcos.
Diferencia máxima entre rankings: 1.082467449010e-15


Posición,Identificador,Título,Puntuación,Fragmento
1,bible-kjv.txt,Bible Kjv,0.099922,"ake man in our image, after our likeness: and let them have dominion over the fish of the sea, and over the fowl of the air, and over the cattle, and over all the earth, and over ever..."
2,blake-poems.txt,Blake Poems,0.099050,"at, And the raven his nest has made In its thickest shade. The gods of the earth and sea Sought through nature to find this tree, But their search was all in vain: There grows..."
3,bryant-stories.txt,Bryant Stories,0.084601,"s and a tail. THE CLOUD One hot summer morning a little Cloud rose out of the sea and floated lightly and happily across the blue sky. Far below lay the earth, brown, dr..."
4,austen-persuasion.txt,Austen Persuasion,0.074344,"all not easily forget Admiral Baldwin. I never saw quite so wretched an example of what a sea-faring life can do; but to a degree, I know it is the same with them all: they are all k..."
5,chesterton-ball.txt,Chesterton Ball,0.067378,"ear to them what seemed to be the upper part of a huge, dark orb or sphere, islanded in a sea of cloud. The Professor's eyes were blazing like a maniac's. ""It is a new world,"" he cr..."



Consulta: divine wisdom, god and religious faith
Necesidad: Encontrar contenido relacionado con Dios, la sabiduría divina y la fe religiosa.
Diferencia máxima entre rankings: 1.720845688169e-15


Posición,Identificador,Título,Puntuación,Fragmento
1,bible-kjv.txt,Bible Kjv,0.155298,"them, What deed is this that ye have done? wot ye not that such a man as I can certainly divine? 44:16 And Judah said, What shall we say unto my lord? what shall we speak? or how shal..."
2,blake-poems.txt,Blake Poems,0.149297,"nfant small; Infant smiles are his own smiles; Heaven and earth to peace beguiles. DIVINE IMAGE To Mercy, Pity, Peace, and Love, All pray in their distress, And to these vi..."
3,bryant-stories.txt,Bryant Stories,0.116142,"he was a mother to the motherless; she was a friend to those who had no friends; she had wisdom greater than schools can teach; we will not let her memory go from us."" So they made a s..."
4,austen-persuasion.txt,Austen Persuasion,0.099331,"f for the folly which asked the question. On one other question which perhaps her utmost wisdom might not have prevented, she was soon spared all suspense; for, after the Miss Musgroves..."
5,chesterton-ball.txt,Chesterton Ball,0.099300,"the gallery below the ball Father Michael had found that man who is the noblest and most divine and most lovable of all men, better than all the saints, greater than all the heroes--man..."


Se calculó la similitud coseno para las cuatro consultas y se seleccionaron los cinco documentos más relevantes (Top-5). Se mostraron sus títulos, puntuaciones y fragmentos de texto, comprobando que los resultados coinciden con los obtenidos mediante Scikit-Learn.